# DCF Valuation Engine from Live Filings

Pulls real financial statement data for any US-listed ticker, projects free cash flow, discounts it at WACC, and outputs an intrinsic value per share — with a WACC × terminal-growth sensitivity table.

**Data source:** Yahoo Finance via `yfinance` (fast, reliable financials + market data).
`SEC EDGAR` is included as a secondary/cross-check source for raw XBRL filing data — useful if you want to verify a line item straight from the 10-K instead of trusting an aggregator.

**How to use this notebook:** change `TICKER` in the Setup cell, run all cells top to bottom, and read the output at the bottom. Every assumption (growth rates, margins, WACC inputs) lives in one clearly marked cell so you can stress-test your own view instead of trusting the defaults.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
!pip install yfinance -q

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

TICKER = "AAPL"   # <-- change this to any US-listed ticker


## 1. Pull financial statements and market data

In [ ]:
stock = yf.Ticker(TICKER)

info = stock.info
income_stmt = stock.financials          # annual income statement
balance_sheet = stock.balance_sheet     # annual balance sheet
cashflow = stock.cashflow               # annual cash flow statement

print(f"{info.get('longName', TICKER)}  ({TICKER})")
print(f"Sector: {info.get('sector')}  |  Industry: {info.get('industry')}")
print(f"Current price: {info.get('currentPrice')}")
print(f"Shares outstanding: {info.get('sharesOutstanding'):,}")


In [ ]:
# Quick look at what's available (statements come with most recent year first)
income_stmt.iloc[:, :4]


## 2. (Optional) Cross-check with SEC EDGAR

`yfinance` financials are convenient but occasionally restate or mislabel line items.
This pulls the company's raw XBRL facts straight from EDGAR so you can sanity-check a
specific figure (e.g. total revenue) against the filing itself.

You need the company's 10-digit CIK number — the lookup below finds it from the ticker.


In [ ]:
def get_cik(ticker):
    headers = {"User-Agent": "research-project contact@example.com"}
    resp = requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers)
    data = resp.json()
    for row in data.values():
        if row["ticker"].upper() == ticker.upper():
            return str(row["cik_str"]).zfill(10)
    return None

def get_edgar_fact(cik, tag="Revenues"):
    headers = {"User-Agent": "research-project contact@example.com"}
    url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{tag}.json"
    resp = requests.get(url, headers=headers)
    if resp.status_code != 200:
        return None
    facts = resp.json()["units"]["USD"]
    df = pd.DataFrame(facts)
    df = df[df["form"] == "10-K"].sort_values("end", ascending=False)
    return df[["end", "val", "fy", "fp", "form"]].head(6)

cik = get_cik(TICKER)
print(f"CIK for {TICKER}: {cik}")
if cik:
    edgar_rev = get_edgar_fact(cik, "Revenues")
    if edgar_rev is None or edgar_rev.empty:
        edgar_rev = get_edgar_fact(cik, "RevenueFromContractWithCustomerExcludingAssessedTax")
    edgar_rev


## 3. Build historical unlevered free cash flow

Unlevered FCF = NOPAT + D&A − CapEx − Δ Net Working Capital.
Pulling straight from the cash flow statement avoids re-deriving D&A/CapEx by hand.


In [ ]:
def safe_row(df, keys):
    """Return the first matching row (by possible label variants) as a Series, else NaN row."""
    for k in keys:
        if k in df.index:
            return df.loc[k]
    return pd.Series(dtype=float)

ebit = safe_row(income_stmt, ["EBIT", "Operating Income"])
tax_provision = safe_row(income_stmt, ["Tax Provision"])
pretax_income = safe_row(income_stmt, ["Pretax Income"])
effective_tax_rate = (tax_provision / pretax_income).clip(0, 0.5)

da = safe_row(cashflow, ["Depreciation And Amortization", "Depreciation Amortization Depletion"])
capex = safe_row(cashflow, ["Capital Expenditure"]).abs()
op_cf = safe_row(cashflow, ["Operating Cash Flow", "Total Cash From Operating Activities"])

nopat = ebit * (1 - effective_tax_rate.fillna(effective_tax_rate.mean()))
hist_fcf = nopat + da - capex

hist_table = pd.DataFrame({
    "EBIT": ebit, "Effective Tax Rate": effective_tax_rate, "NOPAT": nopat,
    "D&A": da, "CapEx": capex, "Unlevered FCF": hist_fcf
}).T
hist_table


In [ ]:
# Historical revenue growth and FCF margin — used to sanity-check our forward assumptions
revenue = safe_row(income_stmt, ["Total Revenue"])
revenue_growth_hist = revenue.pct_change(-1).dropna()  # yfinance columns are most-recent-first
fcf_margin_hist = (hist_fcf / revenue).dropna()

print("Historical revenue growth (YoY):")
print(revenue_growth_hist.round(4))
print("\nHistorical FCF margin:")
print(fcf_margin_hist.round(4))


## 4. Forward assumptions (edit these — this is where your judgment goes in)

Keep every assumption in one place so the model is easy to defend and easy to stress-test.
Anchor the early years near the historical average computed above, then taper growth toward
a terminal rate by year 5 (fade-to-terminal is more defensible than a flat growth rate).


In [ ]:
# --- Revenue growth path (5 years), fading toward a mature/terminal rate ---
revenue_growth_path = [0.08, 0.07, 0.06, 0.05, 0.04]     # edit based on your view / historical avg above

# --- Margin assumptions ---
target_ebit_margin   = 0.28     # steady-state operating margin
tax_rate             = 0.21     # statutory or effective, your call
da_pct_revenue       = 0.03
capex_pct_revenue    = 0.035
nwc_pct_rev_change   = 0.08     # incremental NWC as a % of the *change* in revenue

# --- Terminal value ---
terminal_growth = 0.025         # should not exceed long-run GDP/inflation growth

# --- WACC inputs ---
risk_free_rate   = 0.043        # ~10Y Treasury yield
equity_risk_premium = 0.055
beta = info.get('beta', 1.0) or 1.0
cost_of_debt_pretax = 0.05


In [ ]:
cost_of_equity = risk_free_rate + beta * equity_risk_premium

total_debt = safe_row(balance_sheet, ["Total Debt"]).iloc[0] if "Total Debt" in balance_sheet.index else \
             (safe_row(balance_sheet, ["Long Term Debt"]).iloc[0] + safe_row(balance_sheet, ["Current Debt"]).iloc[0])
cash_and_equiv = safe_row(balance_sheet, ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments"]).iloc[0]
net_debt = total_debt - cash_and_equiv

market_cap = info.get('marketCap')
shares_out = info.get('sharesOutstanding')

E = market_cap
D = total_debt
V = E + D
wacc = (E/V)*cost_of_equity + (D/V)*cost_of_debt_pretax*(1 - tax_rate)

print(f"Cost of equity (CAPM): {cost_of_equity:.2%}")
print(f"Cost of debt (after-tax): {cost_of_debt_pretax*(1-tax_rate):.2%}")
print(f"E/V: {E/V:.1%}   D/V: {D/V:.1%}")
print(f"WACC: {wacc:.2%}")
print(f"Net debt: {net_debt:,.0f}")


## 5. Project forward FCF and discount to present value

In [ ]:
base_revenue = revenue.iloc[0]  # most recent actual year
n_years = len(revenue_growth_path)

proj_revenue = []
proj_fcf = []
rev = base_revenue
prev_rev = base_revenue
for g in revenue_growth_path:
    rev = rev * (1 + g)
    ebit_proj = rev * target_ebit_margin
    nopat_proj = ebit_proj * (1 - tax_rate)
    da_proj = rev * da_pct_revenue
    capex_proj = rev * capex_pct_revenue
    delta_nwc = (rev - prev_rev) * nwc_pct_rev_change
    fcf_proj = nopat_proj + da_proj - capex_proj - delta_nwc
    proj_revenue.append(rev)
    proj_fcf.append(fcf_proj)
    prev_rev = rev

proj_fcf = np.array(proj_fcf)
years = [f"Year {i+1}" for i in range(n_years)]

projection_table = pd.DataFrame({"Revenue": proj_revenue, "Unlevered FCF": proj_fcf}, index=years).T
projection_table


In [ ]:
discount_factors = np.array([(1 + wacc) ** -(t + 1) for t in range(n_years)])
pv_fcf = proj_fcf * discount_factors

terminal_value = proj_fcf[-1] * (1 + terminal_growth) / (wacc - terminal_growth)
pv_terminal_value = terminal_value * discount_factors[-1]

enterprise_value = pv_fcf.sum() + pv_terminal_value
equity_value = enterprise_value - net_debt
intrinsic_price_per_share = equity_value / shares_out
current_price = info.get('currentPrice')

print(f"PV of explicit FCF (5yr): {pv_fcf.sum():,.0f}")
print(f"Terminal value: {terminal_value:,.0f}   |   PV of terminal value: {pv_terminal_value:,.0f}")
print(f"Enterprise value: {enterprise_value:,.0f}")
print(f"Less: net debt: {net_debt:,.0f}")
print(f"Equity value: {equity_value:,.0f}")
print(f"\nIntrinsic value per share: {intrinsic_price_per_share:,.2f}")
print(f"Current market price:      {current_price:,.2f}")
print(f"Implied upside/(downside): {(intrinsic_price_per_share/current_price - 1):.1%}")


## 6. Sensitivity: WACC × terminal growth

In [ ]:
def price_at(w, g):
    if w <= g:
        return np.nan
    disc = np.array([(1 + w) ** -(t + 1) for t in range(n_years)])
    pv = (proj_fcf * disc).sum()
    tv = proj_fcf[-1] * (1 + g) / (w - g)
    pv_tv = tv * disc[-1]
    ev = pv + pv_tv
    return (ev - net_debt) / shares_out

wacc_range = np.round(np.arange(wacc - 0.02, wacc + 0.025, 0.01), 4)
tg_range = np.round(np.arange(terminal_growth - 0.01, terminal_growth + 0.0125, 0.005), 4)

sens = pd.DataFrame(index=[f"{w:.1%}" for w in wacc_range], columns=[f"{g:.1%}" for g in tg_range])
for w in wacc_range:
    for g in tg_range:
        sens.loc[f"{w:.1%}", f"{g:.1%}"] = price_at(w, g)
sens = sens.astype(float)
sens.index.name = "WACC"
sens.columns.name = "Terminal growth"
sens.round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sens.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(sens.columns))); ax.set_xticklabels(sens.columns)
ax.set_yticks(range(len(sens.index))); ax.set_yticklabels(sens.index)
ax.set_xlabel("Terminal growth"); ax.set_ylabel("WACC")
ax.set_title(f"{TICKER} — Intrinsic value/share sensitivity")
for i in range(len(sens.index)):
    for j in range(len(sens.columns)):
        val = sens.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="$/share")
plt.tight_layout()
plt.show()


## 7. Summary

- **Intrinsic value/share** is printed in Section 5, alongside current market price and implied upside.
- The **sensitivity table** in Section 6 shows how fragile that number is to your two least-certain inputs — WACC and terminal growth. A wide spread across the grid means the "fair value" headline is doing a lot of assumption-carrying, which is worth flagging in any write-up.
- To stress-test a different view: change `revenue_growth_path`, `target_ebit_margin`, or the WACC inputs in Section 4 and re-run from there — nothing above needs to change.
- The EDGAR cross-check in Section 2 is there so you can verify the revenue figure against the actual 10-K filing rather than trusting `yfinance`'s aggregation blindly — a habit worth keeping for any number that ends up in front of an IC.
